# LoRA ViT — Train & Compare
Trains multiple LoRA configs on Food-101, saves checkpoints, then compares them side by side.

**Before running:** Runtime → Change runtime type → T4 GPU

## 1. Install deps (then Runtime → Restart runtime)

In [ ]:
!pip install -q --force-reinstall \
    'jsonargparse[signatures]==4.23.1' \
    'peft>=0.4.0' \
    'transformers>=4.30.0' \
    'torchmetrics>=1.0.0'
print('Done — now Restart runtime, then continue from cell 3.')

## 2. Clone repos & upload scripts

In [ ]:
import os

if not os.path.exists('/content/peft-vit'):
    !git clone https://github.com/rezaakb/peft-vit /content/peft-vit

if not os.path.exists('/content/lora-vit-food101'):
    !git clone https://github.com/Mtlukasik/lora-vit-food101 /content/lora-vit-food101

# Copy train_lora.py from your repo to /content (or upload manually)
!cp /content/lora-vit-food101/train_lora.py /content/train_lora.py

print('Ready.')
!ls /content/

## 3. Config — edit your runs here

In [ ]:
# ── Define the runs you want to train ────────────────────────────────────────
# Each dict maps directly to train_lora.py CLI args.
# Add, remove or edit entries freely.

RUNS = [
    {
        'rank': 4,
        'epochs': 20,
        'lr': 0.01,
        'batch_size': 64,
        'target_modules': ['query', 'value'],
    },
    {
        'rank': 8,
        'epochs': 20,
        'lr': 0.01,
        'batch_size': 64,
        'target_modules': ['query', 'value'],
    },
    {
        'rank': 16,
        'epochs': 20,
        'lr': 0.01,
        'batch_size': 64,
        'target_modules': ['query', 'value', 'key'],
    },
]

SAVE_DIR = '/content/checkpoints'

import os
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'{len(RUNS)} run(s) configured, saving to {SAVE_DIR}')

## 4. Train all runs

In [ ]:
import subprocess, sys, json, os
from datetime import datetime

def build_cmd(run_cfg, save_dir):
    cmd = [
        sys.executable, '/content/train_lora.py',
        '--rank',       str(run_cfg['rank']),
        '--epochs',     str(run_cfg['epochs']),
        '--lr',         str(run_cfg['lr']),
        '--batch_size', str(run_cfg['batch_size']),
        '--save_dir',   save_dir,
        '--target_modules', *run_cfg['target_modules'],
    ]
    if 'lora_alpha' in run_cfg:
        cmd += ['--lora_alpha', str(run_cfg['lora_alpha'])]
    return cmd

env = os.environ.copy()
env['PYTHONPATH'] = '/content/peft-vit/src:/content/peft-vit'

for i, run_cfg in enumerate(RUNS):
    mods = '_'.join(run_cfg['target_modules'])
    print(f'\n{"="*60}')
    print(f'RUN {i+1}/{len(RUNS)} | r={run_cfg["rank"]} epochs={run_cfg["epochs"]} modules={mods}')
    print(f'{"="*60}')
    start = datetime.now()
    cmd = build_cmd(run_cfg, SAVE_DIR)
    result = subprocess.run(cmd, env=env)
    elapsed = datetime.now() - start
    if result.returncode == 0:
        print(f'\n✓ Run {i+1} done in {elapsed}')
    else:
        print(f'\n✗ Run {i+1} FAILED (exit code {result.returncode})')

print('\nAll runs complete.')

## 5. Compare results

In [ ]:
import json, glob, os
import torch
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader
from transformers import ViTForImageClassification
from peft import LoraConfig, get_peft_model

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

val_tf = T.Compose([
    T.Resize(256), T.CenterCrop(224),
    T.ToTensor(), T.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5])
])
food_ds = torchvision.datasets.Food101('/content/data', split='test', transform=val_tf, download=False)
food_dl = DataLoader(food_ds, batch_size=128, shuffle=False, num_workers=2, pin_memory=True)

def evaluate(model, dl):
    model.eval()
    correct, n = 0, 0
    with torch.no_grad():
        for imgs, labels in dl:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            with torch.cuda.amp.autocast():
                logits = model(imgs).logits
            correct += (logits.argmax(1) == labels).sum().item()
            n += len(labels)
    return 100 * correct / n

def load_model(meta):
    run_dir = os.path.dirname(meta['_path'])
    base = ViTForImageClassification.from_pretrained(
        meta['model_name'], num_labels=meta['num_classes'], ignore_mismatched_sizes=True
    )
    cfg = LoraConfig(
        r=meta['rank'], lora_alpha=meta['lora_alpha'],
        target_modules=meta['target_modules'], lora_dropout=0.1, bias='none'
    )
    model = get_peft_model(base, cfg)
    model.load_state_dict(torch.load(os.path.join(run_dir, 'best.pt'), map_location=DEVICE))
    return model.to(DEVICE)

# Load all meta.json files
meta_files = sorted(glob.glob(f'{SAVE_DIR}/**/meta.json', recursive=True))
if not meta_files:
    print('No trained runs found. Run cell 4 first.')
else:
    results = []
    for path in meta_files:
        with open(path) as f:
            meta = json.load(f)
        meta['_path'] = path
        print(f'Evaluating {meta["run_name"]}...')
        model = load_model(meta)
        acc = evaluate(model, food_dl)
        results.append({'run': meta['run_name'], 'rank': meta['rank'],
                         'alpha': meta['lora_alpha'], 'modules': meta['target_modules'],
                         'epochs': meta['epochs'], 'best_ckpt_acc': meta['best_val_acc'],
                         'food101_acc': acc})
        del model
        torch.cuda.empty_cache()

    results.sort(key=lambda x: x['food101_acc'], reverse=True)

    print(f'\n{"="*72}')
    print(f'{"RUN":<38} {"r":>4} {"modules":<20} {"Food-101":>10}')
    print(f'{"─"*72}')
    for r in results:
        mods = '+'.join(r['modules'])
        print(f'{r["run"]:<38} {r["rank"]:>4} {mods:<20} {r["food101_acc"]:>9.2f}%')
    print(f'{"="*72}')

## 6. Plot training curves

In [ ]:
import json, glob
import matplotlib.pyplot as plt

meta_files = sorted(glob.glob(f'{SAVE_DIR}/**/meta.json', recursive=True))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for path in meta_files:
    with open(path) as f:
        meta = json.load(f)
    label = f'r={meta["rank"]} {"+".join(meta["target_modules"])}'
    epochs = range(1, len(meta['history']['train_loss']) + 1)
    ax1.plot(epochs, meta['history']['train_loss'], marker='o', markersize=3, label=label)
    ax2.plot(epochs, meta['history']['val_acc'],   marker='o', markersize=3, label=label)

ax1.set_title('Train Loss'); ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.legend()
ax2.set_title('Val Accuracy (Food-101)'); ax2.set_xlabel('Epoch'); ax2.set_ylabel('Acc %'); ax2.legend()
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/training_curves.png', dpi=150)
plt.show()
print(f'Saved to {SAVE_DIR}/training_curves.png')

## 7. Save checkpoints to Google Drive (optional)

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r /content/checkpoints /content/drive/MyDrive/lora-vit-food101-checkpoints